In [3]:
import re, glob
from pathlib import Path
from typing import Optional, Dict
import pandas as pd

RUN_ID_RE = re.compile(r"^\d{8}_\d{6}$")  # e.g., 20250910_143637

def find_run_id(path: Path) -> Optional[str]:
    """Walk up parents and return the first dir name that looks like YYYYMMDD_HHMMSS."""
    for part in [path] + list(path.parents):
        if RUN_ID_RE.match(part.name):
            return part.name
    return None

def expand_one(glob_or_dir: str) -> Path:
    p = Path(glob_or_dir)
    if p.is_dir():
        files = sorted(p.glob("*.parquet"))
    else:
        files = sorted(Path(x) for x in glob.glob(glob_or_dir))
    if not files:
        raise FileNotFoundError(f"No parquet files found for: {glob_or_dir}")
    return files[0]  # representative file

def check_run_consistency(parquet_glob_or_dir: str, blockmax_csv: str) -> Dict:
    # --- run-id check ---
    pq_sample = expand_one(parquet_glob_or_dir)
    bm_path = Path(blockmax_csv)
    run_id_pq = find_run_id(pq_sample)
    run_id_bm = find_run_id(bm_path)
    same_run_id = (run_id_pq is not None) and (run_id_pq == run_id_bm)

    # --- filename overlap check ---
    bm = pd.read_csv(bm_path)
    bm_names = set(Path(x).name for x in bm["file_name"].astype(str).dropna().tolist())

    if Path(parquet_glob_or_dir).is_dir():
        pq_files = sorted(Path(parquet_glob_or_dir).glob("*.parquet"))
    else:
        pq_files = sorted(Path(x) for x in glob.glob(parquet_glob_or_dir))
    pq_names = set(p.name for p in pq_files)

    intersection = bm_names & pq_names
    denom = max(1, min(len(bm_names), len(pq_names)))
    overlap_ratio = len(intersection) / denom

    verdict = (same_run_id and overlap_ratio >= 0.6)

    return {
        "parquet_glob_or_dir": str(parquet_glob_or_dir),
        "blockmax_csv": str(bm_path),
        "run_id_parquet": run_id_pq,
        "run_id_blockmax": run_id_bm,
        "same_run_id": same_run_id,
        "n_blockmax_files": len(bm_names),
        "n_parquet_files": len(pq_names),
        "filename_overlap_ratio": overlap_ratio,
        "sample_overlap_names": sorted(list(intersection))[:5],
        "looks_like_same_run": verdict,
    }

# ---- Example call (edit paths) ----
res = check_run_consistency(
    "../20250910_203447/consumer/consumer-sts-0_consumer-result/2025-09-10_18-34/batch_consumer-sts-*.parquet",
   "../20250910_203447/merge/merge-sts-0_merge-metrics/2025-09-10_18-34-48/block_maxima.csv"
)
res


{'parquet_glob_or_dir': '../20250910_203447/consumer/consumer-sts-0_consumer-result/2025-09-10_18-34/batch_consumer-sts-*.parquet',
 'blockmax_csv': '../20250910_203447/merge/merge-sts-0_merge-metrics/2025-09-10_18-34-48/block_maxima.csv',
 'run_id_parquet': '20250910_203447',
 'run_id_blockmax': '20250910_203447',
 'same_run_id': True,
 'n_blockmax_files': 804,
 'n_parquet_files': 804,
 'filename_overlap_ratio': 1.0,
 'sample_overlap_names': ['batch_consumer-sts-0_20250911_003630_607114.parquet',
  'batch_consumer-sts-0_20250911_003817_066150.parquet',
  'batch_consumer-sts-0_20250911_004002_969419.parquet',
  'batch_consumer-sts-0_20250911_004145_542823.parquet',
  'batch_consumer-sts-0_20250911_004329_456614.parquet'],
 'looks_like_same_run': True}